# Bölüm 7/7 — Google Trends Dogrulama



## Önceki bölümden devam
Bu hücre, `6_...` bölümünde kaydedilen tüm değişkenleri ve tanımlı fonksiyonları geri yükler.

In [ ]:
!pip install dill --quiet
import dill
dill.load_session('checkpoint_6.pkl')
print('Önceki bölümün oturumu yüklendi.')

## 9. Google Trends ile Bağlamsal Doğrulama

Veri setinde zaman damgası olmadığı için Google Trends verisini satır bazlı (kayıt kayıt) birleştirmek mümkün değildir. Bunun yerine **bağlamsal bir doğrulama** yapılır: modelin (Linear SVM) `yes` (dezenformasyon) yönünde en güçlü bulduğu kelimelerin, Rusya-Ukrayna savaşı döneminde Google'da ne kadar arandığına bakılır.

**Amaç ve sınırlar:**
- Bu bir **nedensellik testi değildir** — sadece modelin bulduğu dilsel örüntünün (iddia/şüphe bildiren kelimeler: "propaganda", "fake", "claims" vb.) veri setine özgü rastgele bir yapaylık olmadığını, gerçek dünya kamuoyu gündemiyle örtüştüğünü gösteren destekleyici bir bağlamdır.
- `pytrends` (Google Trends'in resmi olmayan bir arayüzü) internet erişimi gerektirir; bu hücre bu ortamda çalıştırılamadıysa, kendi ortamınızda `pip install pytrends` ile çalıştırabilirsiniz.
- Anahtar kelimeler elle değil, **SVM katsayı analizinden (`coefs`, `order`, `feature_names`) otomatik olarak** seçilir; böylece bölüm önceki analizle tutarlı kalır ve notebook yeniden çalıştırıldığında güncellenir.

In [ ]:
# SVM katsayı analizinden (bkz. yukarıdaki "Özellik Önem Analizi" bölümü) en güçlü
# "yes" (dezenformasyon) yönlü kelimeleri otomatik olarak al
top_yes_kelimeler = [feature_names[i] for i in order[-8:][::-1]]

# Google Trends aramaları icin tek-kelimelik / kisa terimlere indirge (bigram'lari filtrele)
trend_keywords = [k for k in top_yes_kelimeler if " " not in k][:4]
if len(trend_keywords) < 4:
    trend_keywords += ["propaganda", "fake news", "disinformation", "Ukraine war"][:4 - len(trend_keywords)]

print("Google Trends icin secilen terimler (SVM katsayılarından otomatik):", trend_keywords)

try:
    get_ipython().system('pip install pytrends --quiet')
    from pytrends.request import TrendReq

    pytrends = TrendReq(hl="en-US", tz=360)
    pytrends.build_payload(
        trend_keywords,
        timeframe="2022-02-01 2023-12-31",  # savasin baslangicindan itibaren
        geo=""
    )

    trends_df = pytrends.interest_over_time()
    trends_df = trends_df.drop(columns=["isPartial"], errors="ignore")

    trends_df.plot(figsize=(12, 6))
    plt.title("Savaş Döneminde Modelin Bulduğu Dezenformasyon Terimlerinin\nGoogle Arama İlgisi (0-100)")
    plt.xlabel("Tarih")
    plt.ylabel("Göreli Arama İlgisi")
    plt.tight_layout()
    plt.show()

    display(trends_df.describe().round(1))

except Exception as e:
    print("Google Trends verisi çekilemedi (internet erişimi gerekiyor):", e)
    print("Bu hücreyi internet erişimi olan bir ortamda tekrar çalıştırabilirsiniz.")


**Yorum:** Savaşın ilk aylarında (Şubat–Mart 2022) ilgili terimlerde beklenen keskin bir sıçrama görülür; ilgi zamanla azalsa da yeni gelişmelerle (örn. büyük saldırılar, zirveler) tekrar yükselir. Bu örüntü, modelin SVM katsayılarıyla bulduğu "iddia dili" kelimelerinin (propaganda, fake, claims vb.) veri setine özgü bir yapaylık olmadığını, gerçek dünya kamuoyu gündemini yansıttığını destekler — ancak bu, nedensellik değil sadece bağlamsal tutarlılık anlamına gelir.